In [ ]:
# ============================================================
# 01_PREPARE_YOLO_DATASET: REAL + SYNTHETIC
# ============================================================

!pip install -q "numpy<2.0" "protobuf~=3.20.0"

import os
import json
import cv2
import random
import yaml
from tqdm import tqdm
import shutil

# ---------------- CONFIG ----------------
BASE_DATASET_DIR = '/kaggle/input/zalo-video/train'
OUTPUT_DIR       = '/kaggle/working/yolo_dataset_doan'

# Thư mục synthetic đã tạo từ script synthetic trước đó
SYNTH_ROOT       = '/kaggle/input/synthetic-dataset/synth_train_rembg'

TRAIN_RATIO      = 0.7
CLASS_ID         = 0
CLASS_NAME       = 'target_object'
# ----------------------------------------


def convert_to_yolo(box, img_w, img_h):
    """
    Converts (x1, y1, x2, y2) absolute pixel coordinates to
    YOLO's (x_center_norm, y_center_norm, width_norm, height_norm) format.
    """
    try:
        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
        img_w, img_h = float(img_w), float(img_h)

        dw = 1.0 / img_w
        dh = 1.0 / img_h

        x_center = (x1 + x2) / 2.0
        y_center = (y1 + y2) / 2.0
        width = x2 - x1
        height = y2 - y1

        x_center_norm = x_center * dw
        y_center_norm = y_center * dh
        width_norm = width * dw
        height_norm = height * dh

        return (x_center_norm, y_center_norm, width_norm, height_norm)
    except Exception as e:
        print(f"Error in coordinate conversion: {e}")
        return None


def process_frame(cap, frame_num, boxes, video_id, split_name, img_w, img_h, output_base_dir):
    """
    Read a frame and save it along with its YOLO annotations.
    """

    output_images_dir = os.path.join(output_base_dir, 'images', split_name)
    output_labels_dir = os.path.join(output_base_dir, 'labels', split_name)

    # Capture the specific frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()

    if not ret:
        print(f" Warning: Frame {frame_num} of video {video_id} could not be read from video.")
        return

    # Specify file paths
    file_basename = f"{video_id}_frame_{frame_num:06d}"
    image_path = os.path.join(output_images_dir, f"{file_basename}.jpg")
    label_path = os.path.join(output_labels_dir, f"{file_basename}.txt")

    # Save images
    cv2.imwrite(image_path, frame)

    # Save YOLO annotations
    with open(label_path, 'w') as f:
        for box in boxes:
            # Convert to YOLO format
            yolo_coords = convert_to_yolo(box, img_w, img_h)

            if yolo_coords:
                x_c, y_c, w, h = yolo_coords
                # Write to label file
                f.write(f"{CLASS_ID} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")


def create_yaml_file(output_dir, class_name):
    """Creates the dataset.yaml file required by YOLO."""

    # Get absolute paths for the YAML file
    train_path = os.path.abspath(os.path.join(output_dir, 'images', 'train'))
    val_path   = os.path.abspath(os.path.join(output_dir, 'images', 'val'))

    yaml_content = {
        'train': train_path,
        'val':   val_path,
        'nc':    1,
        'names': [class_name]
    }

    yaml_path = os.path.join(output_dir, 'dataset.yaml')

    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

    print(f"\n✅ Successfully created {yaml_path}")
    print("This file points to your training and validation data.")


def build_real_yolo_dataset():
    """
    Tạo YOLO dataset từ bộ real (observing/train):
    - /images/train, /images/val
    - /labels/train, /labels/val
    """
    annotations_file = os.path.join(BASE_DATASET_DIR, 'annotations', 'annotations.json')
    samples_dir      = os.path.join(BASE_DATASET_DIR, 'samples')

    if not os.path.exists(annotations_file):
        print(f"Error: Annotation file not found at {annotations_file}")
        return

    if not os.path.exists(samples_dir):
        print(f"Error: Samples directory not found at {samples_dir}")
        return

    # Create all output directories
    os.makedirs(os.path.join(OUTPUT_DIR, 'images', 'train'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'images', 'val'),   exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'labels', 'train'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'labels', 'val'),   exist_ok=True)

    # Load all annotation records
    print("📥 Loading annotations...")
    with open(annotations_file, 'r') as f:
        try:
            all_video_records = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Error: Could not parse annotations.json. Invalid JSON: {e}")
            return

    if not isinstance(all_video_records, list):
        print(f"Error: Expected annotations.json to contain a list of video records.")
        return

    random.seed(42)  # reproducible split

    print("🎥 Processing videos and splitting frames (real data)...")
    for record in tqdm(all_video_records, desc="Processing Videos"):
        video_id   = record['video_id']
        video_path = os.path.join(samples_dir, video_id, 'drone_video.mp4')

        if not os.path.exists(video_path):
            print(f"Warning: Video file not found, skipping: {video_path}")
            continue

        # group boxes by frame
        frames_to_process = {}
        for interval in record.get('annotations', []):
            for bbox_data in interval.get('bboxes', []):
                try:
                    frame_num = int(bbox_data['frame'])
                    box = (
                        int(bbox_data['x1']),
                        int(bbox_data['y1']),
                        int(bbox_data['x2']),
                        int(bbox_data['y2'])
                    )

                    if frame_num not in frames_to_process:
                        frames_to_process[frame_num] = []
                    frames_to_process[frame_num].append(box)
                except KeyError as e:
                    print(f"Warning: Missing key {e} in bbox data for {video_id}, skipping box.")
                except Exception as e:
                    print(f"Warning: Error processing bbox data for {video_id}: {e}, skipping box.")

        if not frames_to_process:
            continue

        # open video, get size
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error: Could not open video {video_path}, skipping.")
            continue

        video_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        if video_width == 0 or video_height == 0:
            print(f"Error: Could not get dimensions for video {video_path}, skipping.")
            cap.release()
            continue

        # split frame-level into train/val
        frame_items = list(frames_to_process.items())
        random.shuffle(frame_items)

        split_index = int(len(frame_items) * TRAIN_RATIO)
        train_frames = frame_items[:split_index]
        val_frames   = frame_items[split_index:]

        for frame_num, boxes in train_frames:
            process_frame(cap, frame_num, boxes, video_id, 'train', video_width, video_height, OUTPUT_DIR)

        for frame_num, boxes in val_frames:
            process_frame(cap, frame_num, boxes, video_id, 'val', video_width, video_height, OUTPUT_DIR)

        cap.release()

    create_yaml_file(OUTPUT_DIR, CLASS_NAME)
    print("\n✅ Real YOLO dataset is ready at:", os.path.abspath(OUTPUT_DIR))


def merge_synth_into_yolo_train():
    """
    Merge synthetic images/labels vào YOLO train:
    - Đọc từ SYNTH_ROOT/<sample>/images/*.jpg
    - Copy vào OUTPUT_DIR/images/train
    - Label YOLO tương ứng copy vào OUTPUT_DIR/labels/train
    """
    if not os.path.exists(SYNTH_ROOT):
        print(f"⚠️ Synthetic root not found: {SYNTH_ROOT} -> skip merge.")
        return

    synth_samples = [
        d for d in os.listdir(SYNTH_ROOT)
        if os.path.isdir(os.path.join(SYNTH_ROOT, d))
    ]

    if not synth_samples:
        print(f"⚠️ No synthetic sample folders under {SYNTH_ROOT}")
        return

    dst_img_dir = os.path.join(OUTPUT_DIR, 'images', 'train')
    dst_lbl_dir = os.path.join(OUTPUT_DIR, 'labels', 'train')

    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    total_copied = 0

    print("🧩 Merging synthetic into YOLO train ...")
    for sample in synth_samples:
        src_img_dir = os.path.join(SYNTH_ROOT, sample, 'images')
        src_lbl_dir = os.path.join(SYNTH_ROOT, sample, 'labels')

        if not os.path.isdir(src_img_dir) or not os.path.isdir(src_lbl_dir):
            continue

        for fname in os.listdir(src_img_dir):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            src_img_path = os.path.join(src_img_dir, fname)
            src_lbl_path = os.path.join(src_lbl_dir, fname.rsplit('.', 1)[0] + '.txt')

            if not os.path.exists(src_lbl_path):
                # synthetic mà không có label thì bỏ qua
                continue

            # Để tránh trùng tên, prefix theo sample
            base_name = f"{sample}__{fname}"
            dst_img_path = os.path.join(dst_img_dir, base_name)
            dst_lbl_path = os.path.join(dst_lbl_dir, base_name.rsplit('.', 1)[0] + '.txt')

            shutil.copy2(src_img_path, dst_img_path)
            shutil.copy2(src_lbl_path, dst_lbl_path)
            total_copied += 1

    print(f"✅ Merged {total_copied} synthetic images into train.")


if __name__ == "__main__":
    print("=== BUILD REAL YOLO DATASET ===")
    build_real_yolo_dataset()

    print("\n=== MERGE SYNTHETIC INTO TRAIN ===")
    merge_synth_into_yolo_train()

    print("\n🎉 DONE PREPARING DATA. YOLO dataset.yaml ở:", os.path.join(OUTPUT_DIR, 'dataset.yaml'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, bu

Processing Videos: 100%|██████████| 14/14 [1:10:41<00:00, 302.93s/it]



✅ Successfully created /kaggle/working/yolo_dataset_doan/dataset.yaml
This file points to your training and validation data.

✅ Real YOLO dataset is ready at: /kaggle/working/yolo_dataset_doan

=== MERGE SYNTHETIC INTO TRAIN ===
🧩 Merging synthetic into YOLO train ...
✅ Merged 4900 synthetic images into train.

🎉 DONE PREPARING DATA. YOLO dataset.yaml ở: /kaggle/working/yolo_dataset_doan/dataset.yaml
